Data Loading & Initial Cleaning


In [1]:
import pandas as pd#data manupulation and analysis

# Load dataset with explicit dtype specification to handle mixed types
data = pd.read_csv("PS_20174392719_1491204439457_log.csv", dtype={'newbalanceOrig': str})  # Treat as string initially

# Convert empty strings to 0 and then to float
data['newbalanceOrig'] = data['newbalanceOrig'].replace(r'^\s*$', '0', regex=True).astype(float)

# Show first few rows
print(data.head())

# Show column names
print(data.columns)
print(f"\nDataset shape: {data.shape}")
print(f"\nData types:\n{data.dtypes}")
print(f"\nFraud distribution:\n{data['isFraud'].value_counts()}")

   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  
Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldba

Data Preprocessing & Feature Engineering


Train-Test Split


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Load dataset with proper type handling
data = pd.read_csv("PS_20174392719_1491204439457_log.csv", dtype={
    'newbalanceOrig': str,
    'type': str
})

# Clean numeric columns
numeric_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
for col in numeric_cols:
    data[col] = data[col].replace(r'^\s*$', '0', regex=True).astype(float)

# Convert transaction types to numerical
data['type'] = data['type'].map({
    'CASH_OUT': 1,
    'PAYMENT': 2,
    'CASH_IN': 3,
    'TRANSFER': 4,
    'DEBIT': 5
}).astype(int)

# Add engineered features
data['balance_change'] = data['oldbalanceOrg'] - data['newbalanceOrig']
data['zero_balance_after'] = (data['newbalanceOrig'] == 0).astype(int)

print("Dataset cleaned & ready for training! 🚀")
print(f"Shape: {data.shape}")
print(f"Fraud distribution:\n{data['isFraud'].value_counts()}")
print(f"Fraud percentage: {(data['isFraud'].sum() / len(data)) * 100:.2f}%")

Dataset cleaned & ready for training! 🚀
Shape: (6362620, 13)
Fraud distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64
Fraud percentage: 0.13%


In [3]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Updated feature set with new features
features = ['type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 
            'oldbalanceDest', 'newbalanceDest', 'balance_change', 'zero_balance_after']

X = data[features]
y = data['isFraud']

# Train-test split with stratification
xtrain, xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train shape: {xtrain.shape}")
print(f"Test shape: {xtest.shape}")
print(f"\nTraining set fraud distribution:\n{ytrain.value_counts()}")
print(f"Test set fraud distribution:\n{ytest.value_counts()}")

# Apply SMOTE to handle class imbalance on training set
print("\n🔧 Applying SMOTE (Synthetic Minority Over-sampling Technique)...")
smote = SMOTE(random_state=42)
xtrain_smote, ytrain_smote = smote.fit_resample(xtrain, ytrain)

print(f"\n✅ After SMOTE:")
print(f"Training set shape: {xtrain_smote.shape}")
print(f"Fraud distribution after SMOTE:\n{pd.Series(ytrain_smote).value_counts()}")
print(f"Class ratio - Legit:Fraud = 1:{(ytrain_smote.sum() / (len(ytrain_smote) - ytrain_smote.sum())):.2f}")

Train shape: (5090096, 8)
Test shape: (1272524, 8)

Training set fraud distribution:
isFraud
0    5083526
1       6570
Name: count, dtype: int64
Test set fraud distribution:
isFraud
0    1270881
1       1643
Name: count, dtype: int64

🔧 Applying SMOTE (Synthetic Minority Over-sampling Technique)...

✅ After SMOTE:
Training set shape: (10167052, 8)
Fraud distribution after SMOTE:
isFraud
0    5083526
1    5083526
Name: count, dtype: int64
Class ratio - Legit:Fraud = 1:1.00


Model Training (Random Forest)


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Initialize and train model with optimized parameters
print("🤖 Training RandomForest with SMOTE-balanced data...")
model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'  # Additional class weighting for robustness
)

# Train the model on SMOTE-balanced data
model.fit(xtrain_smote, ytrain_smote)

# Evaluate the model
train_accuracy = model.score(xtrain_smote, ytrain_smote)
test_accuracy = model.score(xtest, ytest)

print(f"\n📊 Model Performance:")
print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Calculate ROC-AUC for better evaluation with imbalanced data
ytest_pred_proba = model.predict_proba(xtest)[:, 1]
roc_auc = roc_auc_score(ytest, ytest_pred_proba)
print(f"Test ROC-AUC Score: {roc_auc:.4f}")
    
print("✅ Model trained successfully!")

🤖 Training RandomForest with SMOTE-balanced data...

📊 Model Performance:
Training Accuracy: 0.9980
Test Accuracy: 0.9973
Test ROC-AUC Score: 0.9995
✅ Model trained successfully!


Model Evaluation


In [6]:
from sklearn.metrics import classification_report, confusion_matrix

# Predict class labels
ypred = model.predict(xtest)

# Predict fraud probabilities
yprob = model.predict_proba(xtest)[:, 1]

print("="*70)
print("MODEL EVALUATION METRICS (on Test Set)")
print("="*70)

# Confusion Matrix
cm = confusion_matrix(ytest, ypred)
print(f"\n📊 Confusion Matrix:")
print(f"    Predicted Legit  Predicted Fraud")
print(f"Actual Legit    {cm[0, 0]:>12}     {cm[0, 1]:>12}")
print(f"Actual Fraud    {cm[1, 0]:>12}     {cm[1, 1]:>12}")

# Calculate additional metrics
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)
sensitivity = tp / (tp + fn)  # Recall for fraud class
precision = tp / (tp + fp)

print(f"\n🎯 Classification Metrics:")
print(f"True Negatives (TN):  {tn:>12}")
print(f"False Positives (FP): {fp:>12}")
print(f"False Negatives (FN): {fn:>12}")
print(f"True Positives (TP):  {tp:>12}")

print(f"\nSensitivity (Fraud Recall): {sensitivity:.4f} ({sensitivity*100:.2f}%)")
print(f"Specificity (Legit Recall): {specificity:.4f} ({specificity*100:.2f}%)")
print(f"Precision (Fraud):          {precision:.4f}")

# Show classification report
print(f"\n📋 Detailed Classification Report:")
print(classification_report(ytest, ypred, target_names=['Legit', 'Fraud']))

# Feature importance
print(f"\n⭐ Top 5 Most Important Features:")
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
print(feature_importance.head())

print(f"\n✅ Model evaluation complete!")

MODEL EVALUATION METRICS (on Test Set)

📊 Confusion Matrix:
    Predicted Legit  Predicted Fraud
Actual Legit         1267498             3383
Actual Fraud              15             1628

🎯 Classification Metrics:
True Negatives (TN):       1267498
False Positives (FP):         3383
False Negatives (FN):           15
True Positives (TP):          1628

Sensitivity (Fraud Recall): 0.9909 (99.09%)
Specificity (Legit Recall): 0.9973 (99.73%)
Precision (Fraud):          0.3249

📋 Detailed Classification Report:
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00   1270881
       Fraud       0.32      0.99      0.49      1643

    accuracy                           1.00   1272524
   macro avg       0.66      0.99      0.74   1272524
weighted avg       1.00      1.00      1.00   1272524


⭐ Top 5 Most Important Features:
          feature  importance
6  balance_change    0.284641
2   oldbalanceOrg    0.216708
1          amount    0.152881
3  n

Fraud Prediction Example


In [7]:
import numpy as np
import pandas as pd

print("🧪 Testing with Sample Transactions\n")

# Test Case 1: Suspicious transaction (likely fraud)
print("=" * 70)
print("Test Case 1: Suspicious Transaction (Balance Emptied)")
print("=" * 70)
transaction_data_1 = {
    'type': 4,               # TRANSFER
    'amount': 181.00,
    'oldbalanceOrg': 181.00,
    'newbalanceOrig': 0.00,
    'oldbalanceDest': 0.00,
    'newbalanceDest': 0.00
}

transaction_data_1['balance_change'] = transaction_data_1['oldbalanceOrg'] - transaction_data_1['newbalanceOrig']
transaction_data_1['zero_balance_after'] = 1 if transaction_data_1['newbalanceOrig'] == 0 else 0

feature_names = [
    'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest', 'balance_change', 'zero_balance_after'
]

input_data_1 = np.array([[
    transaction_data_1['type'],
    transaction_data_1['amount'],
    transaction_data_1['oldbalanceOrg'],
    transaction_data_1['newbalanceOrig'],
    transaction_data_1['oldbalanceDest'],
    transaction_data_1['newbalanceDest'],
    transaction_data_1['balance_change'],
    transaction_data_1['zero_balance_after']
]])

input_df_1 = pd.DataFrame(input_data_1, columns=feature_names)
prediction_1 = model.predict(input_df_1)[0]
probability_1 = model.predict_proba(input_df_1)[0][1]

print(f"Amount: ${transaction_data_1['amount']:.2f}")
print(f"Sender Balance Before: ${transaction_data_1['oldbalanceOrg']:.2f}")
print(f"Sender Balance After: ${transaction_data_1['newbalanceOrig']:.2f}")
print(f"\n🚨 Prediction: {'FRAUD' if prediction_1 == 1 else 'LEGITIMATE'}")
print(f"Fraud Probability: {probability_1 * 100:.2f}%\n")

# Test Case 2: Normal transaction (likely legitimate)
print("=" * 70)
print("Test Case 2: Normal Transaction")
print("=" * 70)
transaction_data_2 = {
    'type': 2,               # PAYMENT
    'amount': 5000.00,
    'oldbalanceOrg': 50000.00,
    'newbalanceOrig': 45000.00,
    'oldbalanceDest': 5000.00,
    'newbalanceDest': 10000.00
}

transaction_data_2['balance_change'] = transaction_data_2['oldbalanceOrg'] - transaction_data_2['newbalanceOrig']
transaction_data_2['zero_balance_after'] = 1 if transaction_data_2['newbalanceOrig'] == 0 else 0

input_data_2 = np.array([[
    transaction_data_2['type'],
    transaction_data_2['amount'],
    transaction_data_2['oldbalanceOrg'],
    transaction_data_2['newbalanceOrig'],
    transaction_data_2['oldbalanceDest'],
    transaction_data_2['newbalanceDest'],
    transaction_data_2['balance_change'],
    transaction_data_2['zero_balance_after']
]])

input_df_2 = pd.DataFrame(input_data_2, columns=feature_names)
prediction_2 = model.predict(input_df_2)[0]
probability_2 = model.predict_proba(input_df_2)[0][1]

print(f"Amount: ${transaction_data_2['amount']:.2f}")
print(f"Sender Balance Before: ${transaction_data_2['oldbalanceOrg']:.2f}")
print(f"Sender Balance After: ${transaction_data_2['newbalanceOrig']:.2f}")
print(f"Recipient Balance Before: ${transaction_data_2['oldbalanceDest']:.2f}")
print(f"Recipient Balance After: ${transaction_data_2['newbalanceDest']:.2f}")
print(f"\n✅ Prediction: {'FRAUD' if prediction_2 == 1 else 'LEGITIMATE'}")
print(f"Fraud Probability: {probability_2 * 100:.2f}%\n")

🧪 Testing with Sample Transactions

Test Case 1: Suspicious Transaction (Balance Emptied)
Amount: $181.00
Sender Balance Before: $181.00
Sender Balance After: $0.00

🚨 Prediction: FRAUD
Fraud Probability: 100.00%

Test Case 2: Normal Transaction
Amount: $5000.00
Sender Balance Before: $50000.00
Sender Balance After: $45000.00
Recipient Balance Before: $5000.00
Recipient Balance After: $10000.00

✅ Prediction: LEGITIMATE
Fraud Probability: 1.00%



Save the Trained Model


In [ ]:
import joblib
import json
from datetime import datetime

# Save the trained model
model_path = "fraud_detection_model.pkl"
joblib.dump(model, model_path)
print(f"✅ Model saved to: {model_path}")

# Save model metadata
metadata = {
    'model_type': 'RandomForestClassifier',
    'n_estimators': 100,
    'features': feature_names,
    'training_date': datetime.now().isoformat(),
    'test_accuracy': float(test_accuracy),
    'roc_auc_score': float(roc_auc),
    'training_samples': len(xtrain_smote),
    'test_samples': len(xtest),
    'class_imbalance_handling': 'SMOTE (Synthetic Minority Over-sampling)',
    'fraud_detection_rate': float(sensitivity),
    'false_positive_rate': 1 - float(specificity)
}

with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

print(f"✅ Metadata saved to: model_metadata.json")
print(f"\n📊 Model Summary:")
print(f"   - Test Accuracy: {test_accuracy:.4f}")
print(f"   - ROC-AUC Score: {roc_auc:.4f}")
print(f"   - Fraud Detection Rate: {sensitivity*100:.2f}%")
print(f"   - False Positive Rate: {(1-specificity)*100:.2f}%")
print(f"\n🚀 Model is ready for deployment!")

✅ Model saved to: fraud_detection_model.pkl
✅ Metadata saved to: model_metadata.json

📊 Model Summary:
   - Test Accuracy: 0.9973
   - ROC-AUC Score: 0.9995
   - Fraud Detection Rate: 99.09%
   - False Positive Rate: 0.27%

🚀 Model is ready for deployment!


: 